In [0]:
# CustomerLake Cache Refresh — Automated Materialization
# Owner: @data-engineer | Task: DE-CACHE-SDP
# Tag: customerlake_project: customerlake
#
# Automates daily refresh of all 30 _cached tables in _metrics schema.
# Uses INSERT OVERWRITE to atomically replace stale data with fresh reads
# from source views, appending materialized_at timestamp per refresh.
#
# Catalog: cdm_tmforum  |  Schema: _metrics
# Schedule: daily via Lakeflow Jobs

from pyspark.sql import functions as F
from datetime import datetime
import time

In [0]:
# Complete mapping: cached_table_name -> source_fqn
# Standard pattern: customerlake_X_cached reads from cdm_tmforum._metrics.customerlake_X
# Special cases for exec_summary (different source name) and incrementality_report (gold schema)

CACHE_MAP = {
    "customerlake_acquisition_efficiency_cached":          "cdm_tmforum._metrics.customerlake_acquisition_efficiency",
    "customerlake_campaign_performance_bulletproof_cached": "cdm_tmforum._metrics.customerlake_campaign_performance_bulletproof",
    "customerlake_campaign_performance_credible_cached":    "cdm_tmforum._metrics.customerlake_campaign_performance_credible",
    "customerlake_channel_allocation_waste_cached":         "cdm_tmforum._metrics.customerlake_channel_allocation_waste",
    "customerlake_channel_maturity_status_cached":          "cdm_tmforum._metrics.customerlake_channel_maturity_status",
    "customerlake_channel_retention_curves_cached":         "cdm_tmforum._metrics.customerlake_channel_retention_curves",
    "customerlake_closed_loop_cycle_cached":                "cdm_tmforum._metrics.customerlake_closed_loop_cycle",
    "customerlake_cost_to_serve_cached":                    "cdm_tmforum._metrics.customerlake_cost_to_serve",
    "customerlake_counterfactual_performance_cached":       "cdm_tmforum._metrics.customerlake_counterfactual_performance",
    "customerlake_dark_audience_opportunity_cached":        "cdm_tmforum._metrics.customerlake_dark_audience_opportunity",
    "customerlake_exec_summary_cached":                     "cdm_tmforum._metrics.customerlake_executive_summary",
    "customerlake_exec_summary_v2_cached":                  "cdm_tmforum._metrics.customerlake_executive_summary_v2",
    "customerlake_feedback_loop_plan_cached":               "cdm_tmforum._metrics.customerlake_feedback_loop_plan",
    "customerlake_holdout_quality_assessment_cached":       "cdm_tmforum._metrics.customerlake_holdout_quality_assessment",
    "customerlake_incrementality_by_risk_tier_cached":      "cdm_tmforum._metrics.customerlake_incrementality_by_risk_tier",
    "customerlake_incrementality_report_cached":            "cdm_tmforum.gold.incrementality_report",
    "customerlake_incrementality_roi_hero_cached":          "cdm_tmforum._metrics.customerlake_incrementality_roi_hero",
    "customerlake_ltv_cac_fully_loaded_cached":             "cdm_tmforum._metrics.customerlake_ltv_cac_fully_loaded",
    "customerlake_ltv_cac_incremental_cached":              "cdm_tmforum._metrics.customerlake_ltv_cac_incremental",
    "customerlake_ltv_cac_reconciliation_cached":           "cdm_tmforum._metrics.customerlake_ltv_cac_reconciliation",
    "customerlake_ltv_closedloop_validation_cached":        "cdm_tmforum._metrics.customerlake_ltv_closedloop_validation",
    "customerlake_ltv_validation_backtest_cached":          "cdm_tmforum._metrics.customerlake_ltv_validation_backtest",
    "customerlake_measurement_maturity_cached":             "cdm_tmforum._metrics.customerlake_measurement_maturity",
    "customerlake_measurement_maturity_roadmap_cached":     "cdm_tmforum._metrics.customerlake_measurement_maturity_roadmap",
    "customerlake_reallocation_scenario_model_cached":      "cdm_tmforum._metrics.customerlake_reallocation_scenario_model",
    "customerlake_risk_based_targeting_cached":             "cdm_tmforum._metrics.customerlake_risk_based_targeting",
    "customerlake_roi_confidence_analysis_cached":          "cdm_tmforum._metrics.customerlake_roi_confidence_analysis",
    "customerlake_statistical_quality_narrative_cached":    "cdm_tmforum._metrics.customerlake_statistical_quality_narrative",
    "customerlake_statistical_significance_cached":         "cdm_tmforum._metrics.customerlake_statistical_significance",
    "customerlake_suppression_90day_pnl_cached":            "cdm_tmforum._metrics.customerlake_suppression_90day_pnl",
}

In [0]:
# Refresh all 30 _cached tables via INSERT OVERWRITE.
# Each table is atomically replaced with fresh data from its source view
# plus a materialized_at timestamp. Failures on individual tables are
# logged but do not abort the run — partial success is preferred over
# total failure so dashboards keep serving stale-but-present data.

CAT = "cdm_tmforum"
SCHEMA = "_metrics"
run_ts = datetime.utcnow().isoformat()

successes = []
failures = []

for cached_name, source_fqn in CACHE_MAP.items():
    target_fqn = f"{CAT}.{SCHEMA}.{cached_name}"
    t0 = time.time()
    try:
        source_df = spark.read.table(source_fqn)
        # Check if materialized_at already exists in source; if so, overwrite it
        if "materialized_at" in source_df.columns:
            refreshed_df = source_df.drop("materialized_at").withColumn(
                "materialized_at", F.current_timestamp()
            )
        else:
            refreshed_df = source_df.withColumn(
                "materialized_at", F.current_timestamp()
            )
        refreshed_df.write.mode("overwrite").option(
            "overwriteSchema", "true"
        ).saveAsTable(target_fqn)
        elapsed = round(time.time() - t0, 1)
        successes.append((cached_name, elapsed))
        print(f"  OK  {cached_name} ({elapsed}s)")
    except Exception as e:
        elapsed = round(time.time() - t0, 1)
        failures.append((cached_name, str(e)[:200]))
        print(f"  FAIL {cached_name} ({elapsed}s): {str(e)[:120]}")

print(f"\n--- Refresh complete: {len(successes)} OK, {len(failures)} FAIL ---")
if failures:
    print("Failures:")
    for name, err in failures:
        print(f"  {name}: {err}")

In [0]:
# Ensure every refreshed table carries the project tag.
# Also verify the tag is set (idempotent ALTER).

for cached_name, _ in CACHE_MAP.items():
    target_fqn = f"{CAT}.{SCHEMA}.{cached_name}"
    try:
        spark.sql(f"ALTER TABLE {target_fqn} SET TBLPROPERTIES ('customerlake_project' = 'customerlake')")
    except Exception:
        pass  # table may not exist if refresh failed

print(f"\nCache refresh run completed at {run_ts}")
print(f"  Tables refreshed: {len(successes)}/{len(CACHE_MAP)}")
if failures:
    raise Exception(f"{len(failures)} tables failed to refresh: {[f[0] for f in failures]}")